In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import utils.helpers as hlp
from samplers.linear_models import BayesianRegression

In [ ]:
seed = 42
rng = np.random.default_rng(seed)

In [ ]:
def data_generator(
    num_data=1000, beta_star=None,
    sigma=0.5, seed=seed
):
    """ 
    X: (num_data, dim), y: (num_data,)
    beta_star: (dim,)
    """
    beta_star = np.asarray(beta_star)
    dim = beta_star.size
    rng = np.random.default_rng(seed)
    X = rng.standard_normal(size=(num_data, dim))
    y = X @ beta_star + sigma * rng.standard_normal(size=num_data)
    return X, y

# 2D Experiments With Synthetic Data

In [ ]:
# True Parameters 
beta_star = [1.0, 1.0]
# Generate Data
X, y = data_generator(
    num_data=10000, beta_star=beta_star, sigma=0.5
)

## Ordinary Least Squares Estimates

In [ ]:
beta_ols = np.linalg.lstsq(X, y, rcond=None)[0]
print("OLS estimate:", beta_ols)

## Prior Distributions  

We consider the prior distribution of the regression coefficients $\mathbf{\beta}=[\beta_0, \beta_1]^{\top}$ is the uniform distribution over the $\ell_p$ ball,  

$B_{O}(0,s)=\{\beta\in\mathbb{R}^2: \|\mathbf{\beta}\|_{\ell_p}\le s\}$. 

where, $s=t\|\beta^{OLS}\|_p$ for $t\in[0,1]$ is the boundary. 

[Code accepts only $p=1,2$]  

### $\ell_2$ Prior with $s=\frac45\beta^{OLS}$  

In [ ]:
lp = 2
s = 0.8*np.linalg.norm(beta_ols, ord=lp)
prior_dist = hlp.priors(dim=2, N=5000, s=s, lp=lp, rng=rng)
beta_x=prior_dist.T[0]
beta_y=prior_dist.T[1]

X1 = np.linspace(-1.5, 1.5, 1000)
Y = np.linspace(-1.5, 1.5, 1000)
xx, yy = np.meshgrid(X1, Y)
zz = np.power(
    np.power(np.abs(xx), lp) + np.power(np.abs(yy), lp), 1/lp
)
fig = plt.figure(figsize=(8, 8))
plt.contour(X1, Y, zz, [s])
sns.kdeplot(
    x=beta_x, y=beta_y, fill=True, levels=10
)
plt.scatter(beta_star[0], beta_star[1], c='r', s=200, marker=(5,1))
plt.tick_params(labelsize=30)
plt.savefig('images/prior_linreg_lp.png')
plt.show()

## Sampling from the posterior distribution

In [ ]:
def regression_run(
    X=X, y=y, N=50, batch_size=50, eta=5e-5,
    gamma=1e-2, lp=lp, s=s, n_iteration=100,
    size_w=8, sigma=0.5, seed=42,
    nets=("fcn", "cn", "sn", "fdn"),
    figsize=(32, 16)
):
    fig, axes = plt.subplots(2, 4, figsize=figsize, sharey=False)
    network_names = ["Fully Connected", "Circular", "Star", "Disconnected"]
    
    for j, (net, name) in enumerate(zip(nets, network_names)):
        ax_top = axes[0, j]
        ax_bottom = axes[1, j]
        model = BayesianRegression(
            X=X, y=y, N=N, batch_size=batch_size,
            eta=eta, gamma=gamma, lp=lp, s=s,
            n_iteration=n_iteration,
            size_w=size_w, sigma=sigma,
            net=net, seed=seed, type='linear'
        )
        history_all, beta_mean_all = model.sample_parameters(
            method='dpsgld'
        )
        X1 = np.linspace(-1.5, 1.5, 1000)
        Y1 = np.linspace(-1.5, 1.5, 1000)
        xx, yy = np.meshgrid(X1, Y1)
        zz = np.power(
            np.power(np.abs(xx), lp) + np.power(np.abs(yy), lp), 1/lp
        )
        ax_top.contour(X1, Y1, zz, [s])
        agent = rng.integers(0, size_w)
        sns.kdeplot(
            x=history_all[-1,agent,0,:],
            y=history_all[-1,agent,1,:],
            fill=True, levels=10, ax=ax_top
        )
        ax_top.scatter(
            beta_star[0], beta_star[1],
            c='r', s=200, marker=(5,1)
        )
        ax_top.set_title(f"{name}\nAgent {agent}", fontsize=30)
        ax_top.tick_params(axis="both", labelsize=28)
        
        ax_bottom.contour(X1, Y1, zz, [s])
        sns.kdeplot(
            x=beta_mean_all[-1, 0, :],
            y=beta_mean_all[-1, 1, :],
            fill=True, levels=10, ax=ax_bottom
        )
        ax_bottom.scatter(
            beta_star[0], beta_star[1],
            c='r', s=200, label='OLS', marker=(5,1)
        )
        ax_bottom.set_title(f"{name}\nMean", fontsize=30)
        ax_bottom.tick_params(axis="both", labelsize=28)
    plt.tight_layout()
    plt.savefig('images/regression2dsample.png')
    plt.show()

In [ ]:
lp = 2
s = 0.8*np.linalg.norm(beta_ols, ord=lp)
regression_run(
    X=X, y=y, N=100, n_iteration=5000, lp=lp, s=s, batch_size=100,
    gamma=1e-3, sigma=0.5, eta=5e-4, size_w=10
)